In [0]:
%sql
CREATE TABLE demo_employees
(id INT, name STRING, salary DOUBLE);
INSERT INTO demo_employees VALUES
  (1, 'Bill', 100000),
  (2, 'Mary', 120000),
  (3, 'John', 150000),
  (4, 'Sue', 140000);

In [0]:
SELECT * FROM demo_employees;
SELECT * FROM demo_employees LIMIT 1

In [0]:
%sql
SHOW TABLES; --all the tables under the current schema
SHOW CREATE TABLE demo_employees; --ddl
SHOW COLUMNS FROM demo_employees; --columns

In [0]:
%sql
DESCRIBE demo_employees; --columns + datatype
DESCRIBE EXTENDED demo_employees; --columns + table properties
DESCRIBE DETAIL demo_employees; --columns + table properties + table location
DESCRIBE FORMATTED demo_employees; --columns + table properties + table location + table stats

DESCRIBE HISTORY demo_employees; --table history

In [0]:
%sql
UPDATE demo_employees SET name = 'Mary Jane' WHERE id = 2;
    
SELECT * FROM demo_employees;
    
-- DROP TABLE demo_employees;

In [0]:
DESCRIBE HISTORY demo_employees

In [0]:
SELECT * FROM demo_employees TIMESTAMP AS OF "2026-06-11T03:48:04.000+00:00";
SELECT * FROM demo_employees VERSION AS OF 2;

In [0]:
RESTORE TABLE demo_employees TO VERSION AS OF 2;
SELECT * FROM demo_employees;

In [0]:
-- Improves IO operation
OPTIMIZE demo_employees;

-- To set file size
ALTER TABLE demo_employees 
SET TBLPROPERTIES ('delta.targetFileSize' = '128MB');

--This property controls the target size for data files when OPTIMIZE runs (whether manually or via Predictive Optimization). It tells Delta Lake: "when compacting small files, aim for files around 128MB in size."

--Default value: 1GB (for most workloads)

--When to adjust:

---- Smaller files (32-128MB): For tables with many small queries or selective filters
---- Larger files (512MB-1GB+): For large scan queries or ETL workloads


In [0]:
-- Cleans up the old data files (default RETAIN is 7 days)
VACUUM demo_employees;

-- ***OPTIONS:
-- Dry run to see what would be deleted
VACUUM demo_employees DRY RUN;

-- FULL mode (default) - scans all files in table directory
VACUUM demo_employees FULL;

-- LITE mode (Runtime 16.1+) - uses transaction log only
VACUUM demo_employees LITE;

Important Notes
⚠️ Minimum retention: 7 days recommended — Shorter periods risk deleting uncommitted files from long-running jobs

⚠️ Time travel impact — VACUUM removes old data files, preventing time travel beyond the retention period

In [0]:
--The RETAIN clause is deprecated. Instead, use the table property:

-- Set retention duration (default: 7 days)
ALTER TABLE demo_employees 
SET TBLPROPERTIES ('delta.deletedFileRetentionDuration' = '30 days');

-- Then run VACUUM (uses the table property)
VACUUM demo_employees;

-- ** Supported retention formats:

-- '7 days' (default)
-- '30 days'
-- '168 hours'
-- 'interval 14 days'

In [0]:
-- data gets reordered manually based on the key column (full rewrite)
-- Kind of Indexing based on key value which helps in data skipping (no need to read all the files)
OPTIMIZE demo_employees ZORDER BY id

In [0]:
-- Incremental, self-tuning, and more efficient than ZORDER BY
-- Two types:
-- 1. CLUSTER BY AUTO - Databricks automatically selects optimal clustering keys
-- 2. CLUSTER BY (columns) - You specify clustering columns explicitly

-- Add automatic clustering to existing table
ALTER TABLE demo_employees CLUSTER BY AUTO; --(requires predictive optimization enabled in UC)

-- Or add manual clustering with specific columns
ALTER TABLE demo_employees CLUSTER BY (id, name);

-- For new tables, specify during creation:
-- CREATE TABLE new_table (...) CLUSTER BY AUTO;
-- CREATE TABLE new_table (...) CLUSTER BY (column1, column2);

-- After clustering is configured, OPTIMIZE automatically maintains layout
OPTIMIZE demo_employees;

In [0]:
--What Predictive Optimization does:

-- 1.Automatically runs OPTIMIZE on tables
-- 2.Automatically runs VACUUM to clean old files
-- 3.Automatically collects statistics
-- 4.Uses delta.targetFileSize (if set) to determine file sizes during compaction
-- 5. Applies only to Unity Catalog managed tables 


-- Enable at catalog level (all schemas and tables in this catalog inherit)
ALTER CATALOG workspace ENABLE PREDICTIVE OPTIMIZATION;

-- Enable at schema level (all tables in this schema inherit)
ALTER SCHEMA workspace.default ENABLE PREDICTIVE OPTIMIZATION;

-- Reset to inherit from parent
ALTER SCHEMA workspace.default INHERIT PREDICTIVE OPTIMIZATION;

-- Disable if needed
ALTER SCHEMA workspace.default DISABLE PREDICTIVE OPTIMIZATION;

In [0]:
DROP TABLE IF EXISTS demo_employees;